# MCP: Model Context Protocol

Ate agora, conectamos ferramentas ao agente definindo funcoes Python decoradas com `@tool`. Isso funciona bem para ferramentas internas, mas tem uma limitacao: cada ferramenta precisa ser implementada dentro do nosso codigo.

O **MCP (Model Context Protocol)** resolve esse problema. Ele e um protocolo aberto que padroniza a forma como agentes se conectam a fontes de dados e ferramentas externas. A analogia mais direta e com o USB: assim como o USB criou um padrao universal para conectar dispositivos a computadores, o MCP cria um padrao universal para conectar agentes a servicos externos.

Um servidor MCP pode expor tres tipos de primitivos:

- **Tools** -- funcoes que o agente pode chamar (equivalente ao `@tool` que ja conhecemos)
- **Resources** -- dados estaticos que o agente pode consultar (arquivos, documentos, informacoes do sistema)
- **Prompts** -- templates de prompt pre-definidos que o servidor disponibiliza

Neste notebook, vamos conectar um agente a um servidor MCP local e depois a um servidor MCP remoto, mostrando que a interface e identica em ambos os casos.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import sys
import asyncio

if sys.platform == "win32":
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

## Servidor MCP local

Vamos comecar conectando a um servidor MCP que roda na nossa propria maquina. O arquivo `resources/servidor_mcp.py` implementa um servidor com ferramentas de calculo financeiro (juros compostos e conversao de moeda), alem de um resource e um prompt template.

Para conectar, usamos o `MultiServerMCPClient` com transporte `stdio`. Isso significa que o cliente inicia o servidor como um subprocesso e se comunica com ele via entrada/saida padrao.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

cliente = MultiServerMCPClient(
    {
        "servidor_local": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/servidor_mcp.py"],
        }
    }
)

## Recuperando ferramentas, recursos e prompts

Com o cliente conectado, podemos recuperar os tres primitivos que o servidor expoe. Note que todas as chamadas sao **assincronas** (`await`), porque a comunicacao com o servidor MCP acontece via I/O.

In [ ]:
tools = await cliente.get_tools()

for t in tools:
    print(t.name)

As tools do MCP sao identicas as tools que criavamos com `@tool`. O agente nao sabe (e nao precisa saber) se a ferramenta veio de um decorador local ou de um servidor MCP.

In [ ]:
resources = await cliente.get_resources("servidor_local")

for r in resources:
    print(r.metadata["uri"], "-", r.as_string())

Resources sao dados estaticos que o servidor disponibiliza. Diferente de tools, eles nao executam logica -- apenas retornam informacoes.

In [ ]:
prompt = await cliente.get_prompt("servidor_local", "prompt")
prompt = prompt[0].content

print(prompt)

O prompt template vem do servidor, nao do nosso codigo. Isso permite que o mesmo servidor distribua tanto as ferramentas quanto as instrucoes de como usa-las.

## Agente com MCP

Agora vamos criar um agente usando as tools e o prompt que recuperamos do servidor MCP. O `create_agent` recebe as tools exatamente como recebia tools locais.

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model="gpt-4.1-nano",
    tools=tools,
    system_prompt=prompt
)

In [ ]:
from langchain.messages import HumanMessage

resposta = await agente.ainvoke(
    {"messages": [HumanMessage(content="Quanto rendem 50 mil reais a 13% ao ano durante 24 meses?")]}
)

print(resposta["messages"][-1].content)

O agente usou a tool de juros compostos que veio do servidor MCP. Do ponto de vista do agente, nao ha diferenca entre uma tool local e uma tool MCP.

In [ ]:
resposta = await agente.ainvoke(
    {"messages": [HumanMessage(content="Converta 1000 dolares para reais.")]}
)

print(resposta["messages"][-1].content)

## Servidor MCP remoto

Ate aqui conectamos a um servidor local via `stdio`. Mas o MCP tambem funciona com servidores remotos, acessados via HTTP. O transporte muda de `stdio` para `streamable_http`, mas a interface para o agente continua identica.

Vamos conectar ao servidor MCP da **Kiwi.com**, uma plataforma de busca de voos que expoe suas ferramentas via MCP.

In [ ]:
cliente_kiwi = MultiServerMCPClient(
    {
        "kiwi": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    }
)

tools_kiwi = await cliente_kiwi.get_tools()

In [ ]:
for t in tools_kiwi:
    print(t.name)

## Agente de viagem

Com as ferramentas da Kiwi, vamos criar um agente que busca voos reais. Vamos adicionar `InMemorySaver` para manter o contexto da conversa.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agente_viagem = create_agent(
    model="gpt-4.1-nano",
    tools=tools_kiwi,
    system_prompt="Voce e um agente de viagens. Nao faca perguntas adicionais, busque direto as opcoes.",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "viagem-1"}}

In [ ]:
resposta = await agente_viagem.ainvoke(
    {"messages": [HumanMessage(content="Busque um voo de Sao Paulo para Lisboa no dia 15 de julho de 2026")]},
    config
)

print(resposta["messages"][-1].content)

O agente usou as ferramentas da Kiwi para buscar voos reais, com precos e horarios atualizados. O codigo e praticamente identico ao que usamos com o servidor local. A unica diferenca esta na configuracao do cliente: `stdio` para servidores locais, `streamable_http` para servidores remotos.

Essa e a forca do MCP: uma interface padronizada que funciona independente de onde o servidor esta rodando. O agente nao precisa saber se a ferramenta esta na mesma maquina ou em um servidor na nuvem.